In [0]:
spark.sql("create schema if not exists workspace.Netflix")
spark.sql("create volume if not exists workspace.Netflix.raw_data")
spark.sql("create volume if not exists workspace.Netflix.checkpoint_dir")

print("create schema and volume successfully")

In [0]:
%sql
-- 1. ตารางมิติหลัก (Main Dimension - รองรับ SCD Type 2)
CREATE TABLE IF NOT EXISTS workspace.netflix.dim_titles_silver (
  show_id STRING NOT NULL,
  type STRING,
  title STRING,
  release_year INT,
  rating STRING,
  duration STRING,
  description STRING,
  hash_key STRING,
  hash_value STRING,
  active_flag BOOLEAN,
  start_date TIMESTAMP,
  end_date TIMESTAMP
) USING DELTA 
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true', 'delta.autoOptimize.autoCompact'='true');

-- 2. กลุ่มตารางมิติย่อยเดี่ยวมาสเตอร์ (4 ตาราง - ห้ามคีย์ซ้ำ)
CREATE TABLE IF NOT EXISTS workspace.netflix.dim_cast_silver (cast_id STRING NOT NULL, cast_name STRING) USING DELTA TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true', 'delta.autoOptimize.autoCompact'='true');
CREATE TABLE IF NOT EXISTS workspace.netflix.dim_directors_silver (director_id STRING NOT NULL, director_name STRING) USING DELTA TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true', 'delta.autoOptimize.autoCompact'='true');
CREATE TABLE IF NOT EXISTS workspace.netflix.dim_countries_silver (country_id STRING NOT NULL, country_name STRING) USING DELTA TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true', 'delta.autoOptimize.autoCompact'='true');
CREATE TABLE IF NOT EXISTS workspace.netflix.dim_categories_silver (category_id STRING NOT NULL, category_name STRING) USING DELTA TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true', 'delta.autoOptimize.autoCompact'='true');

-- 3. กลุ่มตารางสะพาน (4 ตาราง - คีย์ผสมร่วมกับ _sk เพื่อบันทึกประวัติร่วม)
CREATE TABLE IF NOT EXISTS workspace.netflix.bridge_title_cast_silver (show_id STRING NOT NULL, _sk LONG NOT NULL, cast_id STRING NOT NULL) USING DELTA TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true', 'delta.autoOptimize.autoCompact'='true');
CREATE TABLE IF NOT EXISTS workspace.netflix.bridge_title_director_silver (show_id STRING NOT NULL, _sk LONG NOT NULL, director_id STRING NOT NULL) USING DELTA TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true', 'delta.autoOptimize.autoCompact'='true');
CREATE TABLE IF NOT EXISTS workspace.netflix.bridge_title_country_silver (show_id STRING NOT NULL, _sk LONG NOT NULL, country_id STRING NOT NULL) USING DELTA TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true', 'delta.autoOptimize.autoCompact'='true');
CREATE TABLE IF NOT EXISTS workspace.netflix.bridge_title_category_silver (show_id STRING NOT NULL, _sk LONG NOT NULL, category_id STRING NOT NULL) USING DELTA TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true', 'delta.autoOptimize.autoCompact'='true');

In [0]:
def copy_file_to_volume(file_name) -> str:
    current_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    current_path = "/".join(current_path.split("/")[0:-1])
    from_path = f"/Workspace{current_path}/file_to_volume/{file_name}"
    to_path = f"/Volumes/workspace/netflix/raw_data/{file_name}"
    dbutils.fs.cp(from_path, to_path)
    
    #Print text when copy file successfully
    print(f"copy file from {from_path} to {to_path} successfully")

copy_file_to_volume(file_name = "netflix_titles.csv")